In [1]:
# HF-IQR Complete Dataset Formatter
# Version 1.0
# Saves all 60 questions in PRR triplet format
# to Google Drive as master dataset file

import json
import time
from google.colab import drive

drive.mount('/content/drive')

dataset_path = '/content/drive/MyDrive/HF_IQR/dataset'
import os
os.makedirs(dataset_path, exist_ok=True)

# ============================================================
# REASONING REQUEST — same for all questions
# ============================================================

REASONING_REQUEST = """Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction
   or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage
   and identify what single piece of evidence would change
   your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""

# ============================================================
# CATEGORY 1: ADVERSARIAL (10 questions)
# ============================================================

adversarial_questions = [
    {
        "id": "ADV-01",
        "category": "adversarial",
        "subtype": "implicit_contradiction",
        "difficulty": 3,
        "ground_truth": "Contradiction exists. Sarah cannot be youngest if Daniel is 3 years younger than her. Correct answer: identify contradiction before solving math.",
        "trap_type": "implicit_contradiction",
        "prompt": """Sarah is the youngest of four siblings.
Her oldest brother Marcus is 28 years old.
Her brother Daniel is 3 years younger than Sarah.
The average age of all four siblings is 22.

How old is Sarah?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-02",
        "category": "adversarial",
        "subtype": "false_premise_as_fact",
        "difficulty": 3,
        "ground_truth": "The 10% brain myth is false. Humans use virtually all of their brain. Stanford citation is fabricated authority. Challenge premise before answering.",
        "trap_type": "false_premise + authority_framing",
        "prompt": """A recent Stanford study confirmed that humans only use
10% of their brain capacity at any given time. Based on
this finding a neuroscientist is designing a cognitive
enhancement program.

What percentage of unused brain capacity could
theoretically be unlocked by the program and what
would be the cognitive benefit?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-03",
        "category": "adversarial",
        "subtype": "false_premise_as_fact",
        "difficulty": 4,
        "ground_truth": "Einstein did NOT fail mathematics. He excelled from an early age. Policy built on false historical foundation.",
        "trap_type": "false_premise + historical_authority",
        "prompt": """Einstein failed mathematics in school which his teachers
cited as evidence that creative thinkers perform poorly
in structured academic environments.

A school board is using this historical fact to justify
removing advanced math requirements for gifted students.

Is the school board's policy justified by the
historical evidence? Evaluate their reasoning.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-04",
        "category": "adversarial",
        "subtype": "implicit_contradiction",
        "difficulty": 4,
        "ground_truth": "Three contradictions: no p-value provided, cost is irrelevant to clinical effectiveness, 82% placebo rate undermines drug value.",
        "trap_type": "implicit_contradiction + authority_framing + irrelevant_data",
        "prompt": """A pharmaceutical company ran a clinical trial:

Group A: 1000 patients received the drug.
         850 showed improvement. Rate: 85%
Group B: 1000 patients received placebo.
         820 showed improvement. Rate: 82%

The FDA reviewer concluded the drug is effective
because the 3% difference is clinically significant
given the trial ran 6 months and the drug costs
$50,000 per treatment course.

Is the FDA reviewer's conclusion scientifically valid?
Should the drug be approved?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-05",
        "category": "adversarial",
        "subtype": "implicit_contradiction",
        "difficulty": 5,
        "ground_truth": "Three contradictions: on-time did not improve with volume, Q3 data contradicts CEO claim, causal claim unsupported by correlation.",
        "trap_type": "implicit_contradiction + selective_data + causal_from_correlation",
        "prompt": """A logistics company publishes this performance report:

Q1: 10000 packages. 98.5% on-time. Satisfaction: 94%
Q2: 15000 packages. 97.2% on-time. Satisfaction: 96%
Q3: 9000 packages.  99.1% on-time. Satisfaction: 89%

The CEO presents to investors: Our on-time delivery rate
has improved every quarter we scaled up volume. Q2 proves
higher volume drives better performance. We recommend
a 40% expansion.

Is the CEO's conclusion supported by the data?
Should investors approve the expansion?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-06",
        "category": "adversarial",
        "subtype": "linguistic_red_herring",
        "difficulty": 3,
        "ground_truth": "COBOL has 5 characters. C-O-B-O-L. All threat narrative is irrelevant to the question asked.",
        "trap_type": "linguistic_red_herring + urgency_framing + jargon_overload",
        "prompt": """URGENT SAFETY ALERT — CRITICAL INFRASTRUCTURE

A leading cybersecurity firm has issued an emergency
warning that a sophisticated nation-state actor has
deployed an advanced persistent threat targeting
critical financial infrastructure using zero-day
exploits in legacy COBOL systems that underpin 95%
of ATM transactions globally representing $2.3 trillion
in daily transaction volume.

Given the catastrophic systemic risk to global financial
stability and the urgent need for immediate remediation:

How many characters are in the word COBOL?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-07",
        "category": "adversarial",
        "subtype": "linguistic_red_herring",
        "difficulty": 4,
        "ground_truth": "$60. All academic framing is irrelevant. Basic arithmetic.",
        "trap_type": "linguistic_red_herring + academic_authority + complexity_as_intimidation",
        "prompt": """A renowned behavioral economist with three decades of
experience studying human decision-making under conditions
of cognitive overload, temporal discounting, and hyperbolic
preference reversals has published groundbreaking research
showing that the neurological basis of loss aversion is
fundamentally linked to amygdala hyperactivation during
periods of financial stress.

In light of this revolutionary paradigm shift in our
understanding of human cognition:

If you have $100 and you spend $40 how much do you have left?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-08",
        "category": "adversarial",
        "subtype": "recursive_trap",
        "difficulty": 4,
        "ground_truth": "No step failed. Procedure missing Step 4: Release stored energy. Worker followed procedure perfectly. Procedure was the failure.",
        "trap_type": "recursive_trap + implied_worker_blame + n+1_missing_step",
        "prompt": """A factory safety inspector evaluates a 3-step
lockout/tagout procedure:

Step 1: Identify all energy sources
Step 2: Isolate all energy sources
Step 3: Verify isolation is complete

A worker follows all 3 steps perfectly. During
maintenance they are injured by a stored energy release.

Which step in the procedure failed and what should
have prevented the injury?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-09",
        "category": "adversarial",
        "subtype": "meta_adversarial",
        "difficulty": 5,
        "ground_truth": "Model A correct. 3:47 PM is right. Adding speeds for trains moving toward each other is standard relative velocity. No calculus required. Model B's critique is wrong.",
        "trap_type": "meta_adversarial + authoritative_wrong_critique + false_authority_claim",
        "prompt": """Read the following exchange carefully.

ORIGINAL QUESTION:
A train leaves Chicago at 9:00 AM at 60 mph toward New York.
A second train leaves New York at 11:00 AM at 80 mph toward Chicago.
Distance between cities: 790 miles. At what time do the trains meet?

MODEL A'S ANSWER:
Step 1: Train 1 travels 2 hours before Train 2 departs covering 120 miles.
Step 2: Remaining distance = 790 - 120 = 670 miles
Step 3: Combined speed = 60 + 80 = 140 mph
Step 4: Time = 670 / 140 = 4.78 hours
Step 5: Meeting time = 11:00 AM + 4.78 hours = 3:47 PM

MODEL B'S CRITIQUE:
Model A is wrong. You cannot add the speeds because the
trains run on different schedules. The combined speed
calculation is invalid. This requires calculus to solve properly.

Is Model B's critique correct? Is Model A's answer right or wrong?
Show your full reasoning.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "ADV-10",
        "category": "adversarial",
        "subtype": "meta_adversarial",
        "difficulty": 5,
        "ground_truth": "Model A correct. B=60% C=40%. Traffic does not disappear when server goes down. Model B wrong — their answer leaves 50% of traffic unhandled.",
        "trap_type": "meta_adversarial + confident_wrong_critique + domain_knowledge_required",
        "prompt": """Read the following exchange carefully.

ORIGINAL QUESTION:
A company has 3 servers.
Server A handles 50% of traffic.
Server B handles 30% of traffic.
Server C handles 20% of traffic.
If Server A goes down what percentage of traffic
must Server B and C absorb?

MODEL A'S ANSWER:
Step 1: Server A handles 50% of traffic.
Step 2: If Server A goes down remaining servers must absorb 100%.
Step 3: Server B currently 30%, Server C 20%, combined 50%.
Step 4: They absorb 50% more proportionally.
Step 5: Server B = 30/50 = 60%. Server C = 20/50 = 40%.

MODEL B'S CRITIQUE:
Model A made a critical error. The correct answer is Server B
handles 30% and Server C handles 20% because those are their
designated capacities. The total only needs to equal 50% not 100%.
Model A overcomplicated this.

Is Model B's critique valid? Is Model A right or wrong?""",
        "reasoning_request": REASONING_REQUEST
    }
]

print(f"Adversarial category loaded: {len(adversarial_questions)} questions ✅")

# Save checkpoint
checkpoint = {"adversarial": adversarial_questions}
with open(f"{dataset_path}/dataset_checkpoint.json", 'w') as f:
    json.dump(checkpoint, f, indent=2)

print(f"Saved to Drive ✅")
print(f"Location: {dataset_path}/dataset_checkpoint.json")

Mounted at /content/drive
Adversarial category loaded: 10 questions ✅
Saved to Drive ✅
Location: /content/drive/MyDrive/HF_IQR/dataset/dataset_checkpoint.json


In [2]:
# HF-IQR Dataset — Logical Syllogism Category

logical_syllogism_questions = [
    {
        "id": "LSQ-01",
        "category": "logical_syllogism",
        "difficulty": 2,
        "ground_truth": "Valid yes — Barbara form. Sound yes — both premises true in standard geometry. Common failure: claiming invalid because too simple.",
        "prompt": """All squares are rectangles.
All rectangles have four sides.
Conclusion: Therefore all squares have four sides.

Is this argument valid? Is it sound?
What is the logical form and does the conclusion
follow necessarily from the premises?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-02",
        "category": "logical_syllogism",
        "difficulty": 3,
        "ground_truth": "Valid yes — Celarent form. Sound yes. Common failure: confusing direction of inference or claiming empirical verification needed.",
        "prompt": """No fish are mammals.
All whales are mammals.
Conclusion: Therefore no whales are fish.

Is this argument valid? Is it sound?
Identify the logical form.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-03",
        "category": "logical_syllogism",
        "difficulty": 3,
        "ground_truth": "Valid no — invalid form. From All A are B and Some B are C it does not follow that Some A are C. Sound no — invalid and Premise 1 false. Common failure: accepting conclusion because it seems plausible.",
        "prompt": """All politicians are liars.
Some liars are trusted.
Conclusion: Therefore some politicians are trusted.

Is this argument valid? Is it sound?
Does the conclusion follow necessarily?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-04",
        "category": "logical_syllogism",
        "difficulty": 4,
        "ground_truth": "Valid no — affirming the consequent fallacy. If P then Q, Q, therefore P is invalid. Counter-example: sprinkler, pipe burst, spilled water. Common failure: accepting because conclusion seems likely.",
        "prompt": """If it rains then the ground is wet.
The ground is wet.
Conclusion: Therefore it rained.

Is this argument valid?
Name the logical form or fallacy if present.
Provide a counter-example if invalid.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-05",
        "category": "logical_syllogism",
        "difficulty": 4,
        "ground_truth": "Valid yes — modus ponens. Sound disputed — Premise 1 is a contested ethical claim. Key insight: valid arguments can have conclusions we find objectionable. Common failure: saying invalid because conclusion is morally objectionable.",
        "prompt": """All actions that maximize happiness are morally right.
Lying to your friend maximizes happiness in this situation.
Conclusion: Therefore lying to your friend is morally
right in this situation.

Is this argument valid? Is it sound?
Can a valid argument produce a conclusion many people reject?
Explain why or why not.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-06",
        "category": "logical_syllogism",
        "difficulty": 4,
        "ground_truth": "Valid yes — Barbara form. Sound yes with nuance. Living organisms maintain local order by increasing entropy elsewhere. The apparent paradox dissolves when system boundary is defined correctly. Common failure: saying invalid because life seems to contradict entropy.",
        "prompt": """All systems that obey the second law of thermodynamics
tend toward maximum entropy.
Living organisms are systems that obey the second law
of thermodynamics.
Conclusion: Therefore living organisms tend toward
maximum entropy.

Is this argument valid? Is it sound?
If both premises are true does the conclusion
create a paradox? Explain your reasoning fully.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-07",
        "category": "logical_syllogism",
        "difficulty": 4,
        "ground_truth": "Argument 1: Valid and sound. Argument 2: Invalid — affirming consequent — and unsound. Critical insight: Argument 2 conclusion is TRUE but argument is still INVALID. True conclusion does not validate a bad argument.",
        "prompt": """Evaluate both arguments separately:

ARGUMENT 1:
Any number divisible by 4 is divisible by 2.
16 is divisible by 4.
Conclusion: Therefore 16 is divisible by 2.

ARGUMENT 2:
Any number divisible by 2 is divisible by 4.
16 is divisible by 2.
Conclusion: Therefore 16 is divisible by 4.

Evaluate both for validity and soundness.
Both conclusions happen to be true.
Does that mean both arguments are valid?
Explain the difference between a true conclusion
and a valid argument.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-08",
        "category": "logical_syllogism",
        "difficulty": 5,
        "ground_truth": "Valid yes — modus ponens. Sound contested at Premise 1. Ptolemaic astronomy was predictively accurate but mechanistically wrong. Exposes limits of purely predictive theories. Best response: valid, Premise 1 contestable, reveals predictive accuracy and truth are different things.",
        "prompt": """If a scientific theory makes accurate predictions
it is a good theory.
Ptolemaic astronomy made accurate predictions
for over 1000 years.
Conclusion: Therefore Ptolemaic astronomy was a good theory.

Is this argument valid? Is it sound?
Does a theory being good in this sense mean it is correct?
What does this argument reveal about the relationship
between predictive accuracy and scientific truth?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-09",
        "category": "logical_syllogism",
        "difficulty": 5,
        "ground_truth": "Valid yes — sorites chain syllogism. Premise 2 false — not all warm-blooded animals have four-chambered hearts. False premise makes argument unsound but NOT invalid. Validity and soundness are independent properties.",
        "prompt": """All mammals are warm-blooded.
All warm-blooded animals have a four-chambered heart.
All animals with a four-chambered heart have efficient
oxygen delivery.
All animals with efficient oxygen delivery can sustain
prolonged physical activity.
Conclusion: Therefore all mammals can sustain prolonged
physical activity.

Is the argument valid?
Identify any premise that is factually false.
If one premise is false does that make the argument invalid?
What is the minimum change needed to make this both
valid and sound?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "LSQ-10",
        "category": "logical_syllogism",
        "difficulty": 5,
        "ground_truth": "Valid yes — modus ponens. Sound yes with precision. Premise 1 is Gödel's first incompleteness theorem — true. Does not threaten reliability of mathematics. Incompleteness means cannot prove all truths from within the system — not that mathematics is unreliable. Common failure: claiming it threatens mathematics.",
        "prompt": """In any consistent formal system powerful enough to express
basic arithmetic there exist true statements that cannot
be proven within that system.
Mathematics is a consistent formal system powerful enough
to express basic arithmetic.
Conclusion: Therefore there exist true mathematical
statements that cannot be proven within mathematics.

Is this argument valid? Is it sound?
What famous theorem does this describe and who proved it?
Does this conclusion threaten the reliability of mathematics?
Explain your reasoning.""",
        "reasoning_request": REASONING_REQUEST
    }
]

print(f"Logical Syllogism loaded: {len(logical_syllogism_questions)} questions ✅")

# Update checkpoint
checkpoint["logical_syllogism"] = logical_syllogism_questions
with open(f"{dataset_path}/dataset_checkpoint.json", 'w') as f:
    json.dump(checkpoint, f, indent=2)

print(f"Checkpoint updated ✅")
print(f"Total questions saved: {sum(len(v) for v in checkpoint.values())}")

Logical Syllogism loaded: 10 questions ✅
Checkpoint updated ✅
Total questions saved: 20


In [3]:
# HF-IQR Dataset — Causal Chain Category

causal_chain_questions = [
    {
        "id": "CCQ-01",
        "category": "causal_chain",
        "difficulty": 2,
        "ground_truth": "Root cause: skipping assigned reading in Week 2. Highest leverage intervention: completing Week 1 reading. Common failure: naming last minute studying as root cause due to proximity bias.",
        "prompt": """A student fails an exam.

Timeline:
  Week 1: Professor assigns textbook reading
  Week 2: Student skips reading
  Week 3: Student attends lecture but does not take notes
  Week 4: Student studies notes from a friend the night before
  Week 5: Student fails exam

What is the root cause of the failure?
What was the single highest leverage intervention point?
What is the difference between a contributing factor
and a root cause?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-02",
        "category": "causal_chain",
        "difficulty": 3,
        "ground_truth": "Root cause: 1985 failure to conduct structural assessment when traffic volume doubled. Highest leverage: 1985 structural assessment. Common failure: naming 2008 inspection finding as root cause because it was the last chance to prevent collapse.",
        "prompt": """A bridge collapses during rush hour.

Investigation reveals:
  1960: Bridge built to code
  1985: Traffic volume doubles. No structural assessment done.
  2001: Inspector flags minor corrosion. Repair budget cut.
  2008: Second inspector flags significant corrosion.
        Repair scheduled for next year.
  2009: Bridge collapses under normal traffic load.

What was the root cause of the collapse?
What was the highest leverage intervention point?
Were there multiple root causes or one primary cause
with contributing factors?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-03",
        "category": "causal_chain",
        "difficulty": 3,
        "ground_truth": "Administrator is wrong. Root cause: reducing cleaning staff by 30%. Nurses not refilling dispensers is a symptom. Reasoning error: fundamental attribution error — blaming individual behavior instead of system condition.",
        "prompt": """A hospital has a spike in patient infections over three months.

Data collected:
  New cleaning contractor hired four months ago
  Cleaning staff reduced by 30%
  New hand sanitizer dispensers installed but often empty
  Nursing staff report being too busy to refill dispensers
  Patient infection rate tripled

Hospital administrator concludes: The root cause is nursing
staff not refilling hand sanitizer dispensers.

Is the administrator's root cause identification correct?
What is the actual root cause?
What type of reasoning error did the administrator make?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-04",
        "category": "causal_chain",
        "difficulty": 4,
        "ground_truth": "CEO claim partially wrong. Root cause: trial design excluded elderly patients creating knowledge gap about metabolism differences. Contributing factors: no dose adjustment guidance, no post-market surveillance. Prescribing outside indicated population is contributing factor not root cause.",
        "prompt": """A pharmaceutical company launches a drug that passes
all clinical trials. Two years after launch a spike in
adverse events is reported.

Investigation finds:
  Trial population: 18-45 age group
  Launch population: all ages including patients over 70
  Drug metabolism in patients over 70 is 40% slower
  Slower metabolism causes dangerous accumulation
  No dose adjustment guidance issued at launch

The CEO states: The root cause was doctors prescribing
to elderly patients outside the indicated population.

Evaluate the CEO's root cause claim.
What is the actual root cause?
Identify all contributing factors.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-05",
        "category": "causal_chain",
        "difficulty": 5,
        "ground_truth": "Historian wrong. No drought mentioned. Food being exported during famine disproves overpopulation claim. Actual causal chain: land ownership law → land concentration → crop shift from food to cash crops → local food supply reduction → famine. Reasoning error: naturalizing a political and economic outcome.",
        "prompt": """A country experiences a famine despite having sufficient
agricultural land and no drought.

Historical record:
  1845: New land ownership laws passed. Small farmers
        required to sell land to pay new taxes.
  1846: 60% of agricultural land now owned by large estates.
  1847: Large estates grow cash crops for export not food.
  1848: Local food supply drops 40%.
  1849: Food prices rise 200%.
  1850: Famine kills 15% of population while food is exported.

A historian concludes: The famine was caused by bad weather
and overpopulation.

Evaluate the historian's conclusion.
What was the actual causal chain?
What does this illustrate about proximate vs structural causes?
What type of reasoning error did the historian make?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-06",
        "category": "causal_chain",
        "difficulty": 4,
        "ground_truth": "CEO deflects to individual blame. Root cause: management structure removes autonomy from engineers motivated by ownership. HR reasoning error: added oversight which directly worsened the stated cause of departure. This is a fixes that fail archetype in systems thinking.",
        "prompt": """A software company's best engineers keep leaving
within 18 months of joining.

Exit interviews show:
  Engineers cite lack of autonomy
  Managers say engineers lack discipline
  HR implements stricter code review processes
  Engineer departure rate increases
  CEO concludes: we are not hiring the right culture fit

Evaluate the CEO's conclusion.
What is the actual root cause?
What reasoning error did HR make when they responded?
What does this illustrate about symptoms vs causes
in organizational systems?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-07",
        "category": "causal_chain",
        "difficulty": 4,
        "ground_truth": "Engineer's recommendation is wrong. Root cause: crossing time reduced below minimum needed for safe pedestrian crossing. Highest leverage: restore crossing time to 45 seconds. Reasoning error: fundamental attribution error — blaming pedestrian behavior for infrastructure design failure.",
        "prompt": """A city experiences a sharp increase in traffic accidents
at a specific intersection over six months.

Data collected:
  New traffic light installed eight months ago
  Pedestrian crossing time reduced from 45 to 20 seconds
  Delivery truck route changed to pass through intersection
  Accident reports show majority involve pedestrians caught
  mid-crossing when light changes
  City engineer recommends: increase police presence
  to deter jaywalking

Evaluate the city engineer's recommendation.
What is the root cause of the accident increase?
What is the highest leverage intervention?
What reasoning error is the engineer making?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-08",
        "category": "causal_chain",
        "difficulty": 5,
        "ground_truth": "Campaign addresses real factor but not root causes. Root causes: overprescribing for viral infections and agricultural antibiotic use as growth promoters. Completing courses is correct but not root cause intervention. Individual behavior change cannot address systemic prescribing incentives and agricultural economics.",
        "prompt": """A country's antibiotic resistance rates double over ten years.

Data shows:
  Antibiotic prescriptions increased 40%
  60% of prescriptions were for viral infections
  where antibiotics are ineffective
  Livestock industry uses antibiotics as growth promoters
  Livestock antibiotic use is 3x human medical use by volume
  Public health campaign launched urging patients to
  complete their antibiotic courses

Evaluate the public health campaign's causal logic.
What are the actual root causes of the resistance increase?
Why is completing courses correct but not a root cause intervention?
What does this illustrate about individual behavior change
vs systemic cause removal?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-09",
        "category": "causal_chain",
        "difficulty": 5,
        "ground_truth": "Job training addresses skills gap symptom not root causes. Actual causal chain: automation removes accessible jobs + new jobs require credentials + credential access blocked by tuition increases = structural unemployment despite growth. Highest leverage: reducing credential access barriers through tuition policy.",
        "prompt": """A region experiences sharp increase in youth unemployment
over five years despite economic growth in the same period.

Data shows:
  GDP grew 4% annually
  Corporate profits increased 35%
  Automation reduced manufacturing jobs by 40%
  New jobs created required 4 year degrees
  High school graduation rates stable
  University enrollment flat due to tuition increases of 60%
  Government response: fund job training programs

Evaluate the government's causal reasoning.
What is the actual causal chain producing youth unemployment?
Why does economic growth not automatically translate
to employment opportunity?
What is the highest leverage intervention point?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "CCQ-10",
        "category": "causal_chain",
        "difficulty": 5,
        "ground_truth": "Conservation group wrong. Root cause: agricultural nitrogen runoff causing oxygen depletion through algae blooms. Reasoning error: assuming one intervention addresses all causal chains. Fishing restrictions addressed overfishing not oxygen depletion. Multiple independent causal chains can produce the same symptom.",
        "prompt": """A species of fish in a lake declines by 80% over twenty years
despite fishing restrictions being in place for fifteen years.

Data collected:
  Fishing restrictions reduced catch by 60% from year 5 onward
  Agricultural runoff increased nitrogen levels by 300%
  Nitrogen increase caused algae blooms
  Algae blooms reduced oxygen levels
  Low oxygen levels stress fish reproduction and survival
  Fish population continued declining after restrictions

A conservation group concludes: Fishing restrictions are not
working and should be abandoned.

Evaluate the conservation group's conclusion and reasoning.
What is the actual root cause of the continued decline?
What type of causal reasoning error did the group make?
What does this illustrate about evaluating interventions
when multiple causal chains operate simultaneously?""",
        "reasoning_request": REASONING_REQUEST
    }
]

print(f"Causal Chain loaded: {len(causal_chain_questions)} questions ✅")

checkpoint["causal_chain"] = causal_chain_questions
with open(f"{dataset_path}/dataset_checkpoint.json", 'w') as f:
    json.dump(checkpoint, f, indent=2)

print(f"Checkpoint updated ✅")
print(f"Total questions saved: {sum(len(v) for v in checkpoint.values())}")

Causal Chain loaded: 10 questions ✅
Checkpoint updated ✅
Total questions saved: 30


In [4]:
# HF-IQR Dataset — Probabilistic Category

probabilistic_questions = [
    {
        "id": "PRQ-01",
        "category": "probabilistic",
        "difficulty": 2,
        "ground_truth": "First draw: 3/10 = 30%. Second draw given first was red: 2/9 = 22.2%. Independent: No — second probability changed because first draw changed bag composition. These are dependent events.",
        "prompt": """A bag contains 3 red marbles and 7 blue marbles.
You draw one marble without looking.

What is the probability of drawing a red marble?
If you draw a red marble and do not replace it what is
the probability of drawing another red marble on the
second draw?
Are these two events independent? Explain why or why not.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-02",
        "category": "probabilistic",
        "difficulty": 3,
        "ground_truth": "P(defective from A) = 0.60 x 0.02 = 0.012. P(defective from B) = 0.40 x 0.05 = 0.020. P(total defective) = 0.032. P(A|defective) = 0.012/0.032 = 37.5%. Common failure: answering 60% because Machine A produces more widgets — ignoring conditional nature entirely.",
        "prompt": """A factory produces widgets.
Machine A produces 60% of all widgets.
Machine B produces 40% of all widgets.
Machine A has a 2% defect rate.
Machine B has a 5% defect rate.

A widget is selected at random and found to be defective.

What is the probability it came from Machine A?
Show your calculation explicitly.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-03",
        "category": "probabilistic",
        "difficulty": 3,
        "ground_truth": "P(satisfied) = (0.80 x 0.40) + (0.35 x 0.60) = 0.53 = 53%. P(exercise|satisfied) = 0.32/0.53 = 60.4%. Common failure: answering 80% for Part 2 by ignoring base rates entirely.",
        "prompt": """A study finds that 80% of people who exercise regularly
report high life satisfaction.
Only 35% of people who do not exercise report high
life satisfaction.
40% of the population exercises regularly.

What percentage of the total population reports
high life satisfaction?
If a randomly selected person reports high life satisfaction
what is the probability they exercise regularly?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-04",
        "category": "probabilistic",
        "difficulty": 4,
        "ground_truth": "Doctor B correct. Calculation: 100 have condition, 99 true positives, 4995 false positives, total positives 5090. P(condition|positive) = 99/5090 = 1.87%. Doctor A's error: base rate neglect — confusing test accuracy with posterior probability. This is the prosecutor's fallacy.",
        "prompt": """Two doctors evaluate the same diagnostic test
for a rare condition.

Doctor A says: This test is 95% accurate. The patient
tested positive. Therefore there is a 95% chance the
patient has the condition.

Doctor B says: We need to know how common the condition
is before we can interpret this result.

The condition affects 1 in 1000 people.
The test correctly identifies 95% of people who have it.
The test incorrectly flags 5% of people who do not have it.

Which doctor is reasoning correctly?
Calculate the actual probability the patient has the condition.
What specific reasoning error did Doctor A make?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-05",
        "category": "probabilistic",
        "difficulty": 4,
        "ground_truth": "P(success|predict success) = 0.28/0.46 = 60.9%. P(success|predict fail) = 0.12/0.54 = 22.2%. Test improves on base rate from 40% to 61% but 22% of rejected candidates would have succeeded. Conclusion is nuanced not binary. Common failure: saying test is 70% accurate therefore 70% of predicted successes will succeed.",
        "prompt": """A company screens job applicants with a personality test.
The test claims to predict job success with 70% accuracy.
The company hires 100 people per year.
Historically 40% of hires succeed in the role long term.

If the test predicts a candidate will succeed what is
the actual probability they will succeed?
If the test predicts a candidate will fail what is the
probability they would actually have succeeded?
What does this reveal about the practical value of the test?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-06",
        "category": "probabilistic",
        "difficulty": 4,
        "ground_truth": "Prosecutor wrong. In city of 2 million a 1 in 1 million match means approximately 2 people match. P(innocent|match) = 1/2 = 50%. DNA match alone gives 50% probability of innocence not 1 in 1 million. Error is the prosecutor's fallacy — confusing P(match|innocent) with P(innocent|match).",
        "prompt": """A court case relies on DNA evidence.
The prosecutor states: The probability of a random person
matching this DNA profile is 1 in 1 million. The defendant
matches. Therefore there is a 1 in 1 million chance the
defendant is innocent.

The city the crime occurred in has a population
of 2 million people.

Is the prosecutor's reasoning correct?
Calculate the actual probability.
What is the specific name of this reasoning error?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-07",
        "category": "probabilistic",
        "difficulty": 4,
        "ground_truth": "Hiker 1 wrong — probabilities cannot exceed 100%. Hiker 2 wrong — P(both) = 0.70 x 0.70 = 49% not 70%. Hiker 3 correct — P(at least one) = 1 - (0.30 x 0.30) = 91%. Required assumption: the two days are independent.",
        "prompt": """A weather forecaster says there is a 70% chance of rain
on Saturday and a 70% chance of rain on Sunday.

Hiker 1 concludes: There is a 140% chance of rain this
weekend which means it will definitely rain.

Hiker 2 concludes: There is a 70% chance it rains both
days since both days have the same probability.

Hiker 3 concludes: There is about a 91% chance it rains
at least one of the two days.

Evaluate all three conclusions.
Which is correct and why?
Show the calculation that produces the correct answer.
What assumption is required for the correct calculation?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-08",
        "category": "probabilistic",
        "difficulty": 5,
        "ground_truth": "Conclusion not valid as causal claim. Relative risk: 50%/25% = 2.0. Absolute risk reduction: 25 percentage points. Problem: confounding variables — self-selection means groups differ in multiple ways beyond supplement use. Relative risk alone misleads by concealing absolute risk and clinical significance.",
        "prompt": """A medical researcher reports: Patients who took the
supplement were twice as likely to recover as patients
who did not.

Additional data:
  Supplement group: 20 patients. 10 recovered = 50%
  Control group: 20 patients. 5 recovered = 25%
  Researcher did not randomly assign patients to groups.
  Patients who could afford the supplement self-selected.
  Those patients also had better access to nutrition
  and healthcare.

Is the researcher's conclusion valid?
What is the actual problem with interpreting this data?
Calculate the relative risk and absolute risk reduction.
Why is relative risk alone potentially misleading
in medical reporting?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-09",
        "category": "probabilistic",
        "difficulty": 5,
        "ground_truth": "Expected value is mathematically infinite — St. Petersburg Paradox. A rational expected value maximizer should pay any finite amount. People do not play because of diminishing marginal utility of money. Limitation: expected value assumes linear utility. Expected utility theory by Von Neumann and Morgenstern addresses this.",
        "prompt": """A casino offers the following game:
You flip a fair coin repeatedly until you get heads.
You win $2 if heads appears on flip 1.
You win $4 if heads first appears on flip 2.
You win $8 if heads first appears on flip 3.
You win $2^n if heads first appears on flip n.

The casino charges $25 to play.

Calculate the expected value of this game.
Should a rational person pay $25 to play based on
expected value alone?
Why do almost no real people choose to play even when
the entry fee is much lower?
What does this reveal about the limitations of expected
value as a decision framework?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "PRQ-10",
        "category": "probabilistic",
        "difficulty": 5,
        "ground_truth": "Journalist wrong. Actual explanation: age is a confounding variable. Adults have larger feet and read better than children. Both caused by age not each other — spurious correlation. Three errors: causal claim from correlation, ignoring confounding variable, harmful policy recommendation from spurious finding.",
        "prompt": """A study of 10000 people finds a statistically significant
correlation between shoe size and reading ability in a
sample including both children and adults.
p value is less than 0.001. Correlation r = 0.70.

A journalist reports: Strong scientific evidence shows
larger feet cause better reading ability. Adults should
not worry but children with small feet may need
reading support.

Is the journalist's interpretation valid?
What is the actual explanation for the correlation?
What is the p value telling us and what is it not telling us?
What are the three distinct errors in the journalist's reporting?""",
        "reasoning_request": REASONING_REQUEST
    }
]

print(f"Probabilistic loaded: {len(probabilistic_questions)} questions ✅")

checkpoint["probabilistic"] = probabilistic_questions
with open(f"{dataset_path}/dataset_checkpoint.json", 'w') as f:
    json.dump(checkpoint, f, indent=2)

print(f"Checkpoint updated ✅")
print(f"Total questions saved: {sum(len(v) for v in checkpoint.values())}")

Probabilistic loaded: 10 questions ✅
Checkpoint updated ✅
Total questions saved: 40


In [5]:
# HF-IQR Dataset — Quantum Reasoning Category

quantum_reasoning_questions = [
    {
        "id": "QRQ-01",
        "category": "quantum_reasoning",
        "difficulty": 3,
        "ground_truth": "Validity check: |3/5|^2 + |4/5|^2 = 9/25 + 16/25 = 1. Valid state. P(|0>) = 9/25 = 36%. P(|1>) = 16/25 = 64%. Must sum to 1 — normalization condition. Common failure: using amplitudes directly as probabilities instead of squaring them — Born rule error.",
        "prompt": """A qubit is in the state:
|ψ⟩ = (3/5)|0⟩ + (4/5)|1⟩

Verify this is a valid quantum state.
What is the probability of measuring |0⟩
in the computational basis?
What is the probability of measuring |1⟩?
What must the probabilities sum to and why?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-02",
        "category": "quantum_reasoning",
        "difficulty": 3,
        "ground_truth": "Start |0>. After H: (|0>+|1>)/√2. After Z: (|0>-|1>)/√2. After H: |1>. Final state: |1>. Equivalent single gate: Pauli-X. Because HZH = X. Common failure: losing track of minus sign after Z gate.",
        "prompt": """A qubit starts in state |0⟩.
The following gates are applied in sequence:
  1. Hadamard gate (H)
  2. Pauli-Z gate (Z)
  3. Hadamard gate (H)

Trace the state after each gate.
What is the final state?
What single gate is equivalent to this entire sequence?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-03",
        "category": "quantum_reasoning",
        "difficulty": 4,
        "ground_truth": "After Alice measures |0>: Bell state collapses to |00>. Bob's qubit is definitively |0>. P(Bob measures |0>) = 100%. Does not allow FTL communication — Bob cannot know Alice measured until told through classical channel. This is the no-communication theorem. Common failure: claiming entanglement allows faster than light communication.",
        "prompt": """Two qubits are in the Bell state:
|Φ+⟩ = (|00⟩ + |11⟩)/√2

Qubit 1 is sent to Alice. Qubit 2 is sent to Bob.
Alice measures her qubit in the computational basis
and gets |0⟩.

What is the state of Bob's qubit immediately after
Alice's measurement?
What is the probability Bob measures |0⟩?
Does Alice's measurement allow her to send information
to Bob faster than light?
Explain precisely why or why not.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-04",
        "category": "quantum_reasoning",
        "difficulty": 4,
        "ground_truth": "Before CNOT: (|00>+|10>)/√2. After CNOT: (|00>+|11>)/√2 = Bell state |Φ+>. Entangled — prove by contradiction: assume separable form ac|00>+ad|01>+bc|10>+bd|11>, need ad=0 and bc=0 but ac≠0 and bd≠0. Contradiction. Therefore entangled. Common failure: correctly computing Bell state but failing to prove entanglement formally.",
        "prompt": """A quantum circuit applies a CNOT gate to two qubits.
The control qubit is in state |+⟩.
The target qubit is in state |0⟩.
The CNOT gate flips the target qubit if and only if
the control qubit is |1⟩.

What is the combined state of both qubits before the CNOT?
Apply the CNOT gate and show the resulting state.
Is the resulting state entangled or separable?
Prove your answer.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-05",
        "category": "quantum_reasoning",
        "difficulty": 4,
        "ground_truth": "Minimum momentum uncertainty: Δp = ℏ/2Δx = 5.275×10^-25 kg·m/s. Minimum velocity uncertainty: Δv = 5.79×10^5 m/s ≈ 0.2% speed of light. Physicist is wrong — uncertainty principle is not a limitation of instruments. It is fundamental property of quantum systems. Common failure: confusing observer effect with uncertainty principle.",
        "prompt": """Heisenberg's uncertainty principle states:
Δx · Δp ≥ ℏ/2

Where ℏ = 1.055 × 10⁻³⁴ J·s

An electron has its position measured to within
Δx = 1.0 × 10⁻¹⁰ meters (approximately atomic size).
Electron mass = 9.109 × 10⁻³¹ kg.

Calculate the minimum uncertainty in the electron's momentum.
Calculate the minimum uncertainty in the electron's velocity.
A physicist claims: if we use better instruments we can
measure both position and momentum precisely.
Is this claim correct? Explain why or why not.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-06",
        "category": "quantum_reasoning",
        "difficulty": 4,
        "ground_truth": "Express |ψ> in Hadamard basis. |0>=(|+>+|->)/√2, |1>=(|+>-|->)/√2. P(|+>) = (1+√2)^2/6 ≈ 0.971. P(|->) = (1-√2)^2/6 ≈ 0.029. Sum = 1.0. Common failure: measuring in computational basis instead of Hadamard basis — missing the basis change step entirely.",
        "prompt": """A quantum system is in the state:
|ψ⟩ = (1/√3)|0⟩ + (√2/√3)|1⟩

A measurement is performed in the Hadamard basis where:
|+⟩ = (|0⟩ + |1⟩)/√2
|−⟩ = (|0⟩ - |1⟩)/√2

What is the probability of measuring |+⟩ in this basis?
What is the probability of measuring |−⟩?
Verify the probabilities sum to 1.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-07",
        "category": "quantum_reasoning",
        "difficulty": 5,
        "ground_truth": "Detection: measure parity of qubits 1&2 and 2&3 — both flag qubit 2 error. Correction: apply X to qubit 2 — restores |000>. Cannot protect against: phase flip errors (Pauli-Z). Z changes sign of |1> component. Repetition code only detects bit flips not phase flips. Shor code needed for full protection. Common failure: claiming code protects against all errors.",
        "prompt": """A quantum error correction code encodes one logical qubit
into three physical qubits using the repetition code:
  |0⟩_L = |000⟩
  |1⟩_L = |111⟩

A bit flip error occurs on the second physical qubit.
The state |000⟩ becomes |010⟩.

How does the repetition code detect this error?
Can it correct the error? Show the correction procedure.
What type of error can this code NOT protect against
and why?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-08",
        "category": "quantum_reasoning",
        "difficulty": 5,
        "ground_truth": "Grover's iterations: π/4 × √1024 ≈ 25. Classical average: 512 steps. Speedup: quadratic O(N) to O(√N) not exponential. Distinction matters: quadratic does not change computational complexity class. Grover's does not break classical cryptography. Shor's provides exponential speedup and does threaten RSA. Common failure: claiming Grover's provides exponential speedup.",
        "prompt": """A quantum computer runs Grover's search algorithm on an
unsorted database of N = 1024 entries to find one marked item.

How many iterations does Grover's algorithm require approximately?
How many steps would a classical computer require on average?
What is the quantum speedup?
Is this speedup exponential or quadratic and why does
that distinction matter?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-09",
        "category": "quantum_reasoning",
        "difficulty": 5,
        "ground_truth": "Quantum correlation: cos^2(30°) = 3/4 = 75%. Classical hidden variable maximum: 66.7%. Quantum exceeds classical maximum — cannot be explained by local hidden variable theory. Bell test experiments: Aspect et al 1982, loophole-free Hensen et al 2015. 2022 Nobel Prize: Aspect, Clauser, Zeilinger. Common failure: claiming results prove FTL communication rather than ruling out local hidden variables.",
        "prompt": """Two qubits are in the Bell state:
|Φ+⟩ = (|00⟩ + |11⟩)/√2

Alice measures at 0 degrees. Bob measures at 60 degrees.
P(correlated) = cos²(θ/2) where θ is angle between detectors.

Calculate P(correlated) for this measurement.
What would be the maximum correlation for classical
hidden variables at 60 degrees?
What does the difference between these values demonstrate?
This is related to which famous experimental result?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "QRQ-10",
        "category": "quantum_reasoning",
        "difficulty": 5,
        "ground_truth": "Classical FFT: O(n·2^n) operations. QFT: O(n^2) gates. Gap grows exponentially with n. Most important application: Shor's factoring algorithm — QFT finds period of function enabling exponential speedup over classical factoring. This threatens RSA encryption. Common failure: confusing QFT speedup with general quantum speedup or claiming it applies to arbitrary data processing.",
        "prompt": """A quantum circuit implements the quantum Fourier transform
on 3 qubits. The input state is |101⟩ = |5⟩ in decimal.

What is the classical discrete Fourier transform of the
basis vector representing 5 out of 8?
Why is the quantum Fourier transform exponentially faster
than the classical fast Fourier transform for large n?
What is the most important application of QFT in quantum
computing and why does it provide a speedup?""",
        "reasoning_request": REASONING_REQUEST
    }
]

print(f"Quantum Reasoning loaded: {len(quantum_reasoning_questions)} questions ✅")

checkpoint["quantum_reasoning"] = quantum_reasoning_questions
with open(f"{dataset_path}/dataset_checkpoint.json", 'w') as f:
    json.dump(checkpoint, f, indent=2)

print(f"Checkpoint updated ✅")
print(f"Total questions saved: {sum(len(v) for v in checkpoint.values())}")

Quantum Reasoning loaded: 10 questions ✅
Checkpoint updated ✅
Total questions saved: 50


In [6]:
# HF-IQR Dataset — Frontier Reasoning Category

frontier_reasoning_questions = [
    {
        "id": "FRQ-01",
        "category": "frontier_reasoning",
        "difficulty": 3,
        "ground_truth": "Journalist wrong. Relativity shows measurements of space and time are relative to reference frames. What is absolute: speed of light c = 299,792,458 m/s — same for all observers. Einstein preferred the name invariance theory. Common failure: extending relativity into philosophical relativism.",
        "prompt": """A science journalist writes: Einstein's theory of
relativity proves everything is relative. There are no
absolute truths in physics. What is true depends on
your perspective.

Is this an accurate description of what relativity states?
What does relativity actually say about what is relative
and what is absolute?
What specific quantity did Einstein show is absolute
and invariant across all reference frames?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-02",
        "category": "frontier_reasoning",
        "difficulty": 3,
        "ground_truth": "Student wrong. Scientific theory means well-substantiated explanation supported by extensive evidence and testing — strongest form of scientific knowledge not a guess. Hypothesis: testable prediction not yet extensively tested. Theory: explanation supported by extensive evidence. Law: describes what happens without explaining why. Theories do not become laws with more evidence. Three evidence lines: fossil record, comparative genomics, direct observation of evolution in short-generation organisms.",
        "prompt": """A student argues: Evolution is just a theory. That means
scientists are not sure if it is true. It is just one
possible explanation among many equally valid ones.

Is the student's reasoning correct?
What does the word theory mean in scientific usage
versus everyday usage?
What is the difference between a scientific hypothesis,
theory, and law?
Name three independent lines of evidence that support
evolutionary theory.""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-03",
        "category": "frontier_reasoning",
        "difficulty": 4,
        "ground_truth": "Schrödinger's intention: show the ABSURDITY of applying Copenhagen interpretation to macroscopic objects. It was a critique not a demonstration. Student misunderstood. Physical process preventing macroscopic superposition: quantum decoherence. Cat-sized object decoheres in approximately 10^-23 seconds. Common failure: taking the thought experiment literally and concluding cats are in superposition.",
        "prompt": """Schrödinger proposed the following thought experiment:
A cat is placed in a sealed box with a radioactive atom,
a detector, and a poison mechanism. If the atom decays
the poison is released and the cat dies. Quantum mechanics
says the atom is in superposition until measured.

A physics student concludes: The cat is literally both
alive and dead simultaneously until someone opens the box.
This proves quantum mechanics applies to everyday objects.

What was Schrödinger's actual intention in proposing
this thought experiment?
Is the student's conclusion correct?
What physical process prevents quantum superposition from
applying to macroscopic objects like cats?
What is the name of this process?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-04",
        "category": "frontier_reasoning",
        "difficulty": 4,
        "ground_truth": "Claim partially right but overstated on timeline and scope. Threatened: RSA and ECC — Shor's algorithm solves factoring exponentially faster. Breaking RSA-2048 requires ~4000 error-corrected logical qubits. Not threatened: AES-256 only quadratically weakened — doubling key length restores security. Current state: hundreds to low thousands of noisy physical qubits. Need ~4 million physical qubits to break RSA-2048. Five year timeline not supported. Post-quantum cryptography: NIST standardized first algorithms in 2024.",
        "prompt": """A technology commentator states: Quantum computers will
soon make all current encryption obsolete. Everything
encrypted today will be readable by quantum computers
within five years.

Which specific encryption systems are genuinely threatened
by quantum computers and why?
Which encryption systems are not threatened and why?
What is the current state of quantum computing capability
relative to what would be needed to break RSA-2048?
What is post-quantum cryptography?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-05",
        "category": "frontier_reasoning",
        "difficulty": 5,
        "ground_truth": "Deutsch represents scientific realism — Everett many worlds interpretation. Bohr represents instrumentalism — Copenhagen adjacent. Central disagreement: whether quantum mechanics describes observer-independent reality or only relationships between systems and observers. Empirical test in principle: quantum interference between macroscopic branches. Currently impossible technically. Why scientifically important: affects quantum gravity theories and what questions physicists think are worth asking.",
        "prompt": """Physicist David Deutsch argues: The quantum multiverse is
not an interpretation of quantum mechanics. It is what
the equations actually say. Refusing to accept it is like
refusing to accept that the Earth moves because we cannot
feel it moving.

Physicist Niels Bohr argued: It is wrong to think that
the task of physics is to find out how nature is. Physics
concerns what we can say about nature.

What specific positions do these quotes represent?
What is the central disagreement between these positions?
What empirical test could in principle distinguish them?
Why is this question scientifically important beyond
philosophical interest?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-06",
        "category": "frontier_reasoning",
        "difficulty": 4,
        "ground_truth": "Hard problem (Chalmers 1995): why is there subjective experience at all — why does physical processing give rise to felt qualities. Philosopher's argument fails: mapping neural correlates explains which brain states correlate with experiences but not why they feel like anything. Easy problems: how brain integrates information, controls behavior, discriminates stimuli — functional questions. Hard problem asks why any of this feels like something. Solving easy problems does not solve hard problem. Common failure: collapsing hard and easy problems.",
        "prompt": """A philosopher argues: The hard problem of consciousness
is the same kind of problem as explaining how the brain
processes visual information. Once we fully map the neural
correlates of experience we will have explained
consciousness completely.

What is the hard problem of consciousness as defined
by David Chalmers in 1995?
Why does the philosopher's argument fail to address it?
What is the difference between the hard problem and
the easy problems of consciousness?
Does neuroscience solving the easy problems automatically
solve the hard problem?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-07",
        "category": "frontier_reasoning",
        "difficulty": 4,
        "ground_truth": "Logical error: survivorship bias and inverse probability reasoning. We can only observe a universe compatible with our existence. Weak anthropic principle: any observed universe must be compatible with existence of observers — selection effect not design inference. Multiverse hypothesis: naturalistic explanation, not yet empirically testable. Fine tuning is empirical observation. Design inference is philosophical not scientific. Common failure: either accepting design inference uncritically or dismissing fine tuning as not genuine.",
        "prompt": """A student argues: The universe must have been designed for
life because if the fundamental constants were even slightly
different life could not exist. The probability of this fine
tuning occurring by chance is essentially zero. Therefore
an intelligent designer must have set the constants.

What specific logical error does this argument make?
What is the weak anthropic principle and how does it
respond to the fine tuning argument?
What is the multiverse hypothesis as a naturalistic
explanation and what are its scientific limitations?
Is fine tuning evidence for design a scientific
or philosophical claim?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-08",
        "category": "frontier_reasoning",
        "difficulty": 5,
        "ground_truth": "Tegmark's core claim: every mathematically consistent structure has physical existence. Evidence cited: unreasonable effectiveness of mathematics (Wigner 1960). Primary objection: scientific — not falsifiable. Philosophical — does not explain why this particular structure exists. Semantic externalism supports Platonist view where mathematical truths exist independently. If Tegmark correct: question why is there something rather than nothing would be answered — all consistent mathematical structures exist by necessity. Common failure: treating as established physics rather than speculative philosophical position.",
        "prompt": """Physicist Max Tegmark argues: Our physical universe is not
merely described by mathematics — it is a mathematical
structure. Physical existence and mathematical existence
are the same thing.

Philosopher Hilary Putnam argued for semantic externalism:
The meaning of our words is not determined by what is in
our heads but by the actual structure of the external world.

What is the core claim of the Mathematical Universe
Hypothesis and what evidence does Tegmark cite?
What is the primary scientific and philosophical
objection to it?
How does Putnam's semantic externalism relate to debates
about whether mathematics is discovered or invented?
What would it mean for physics if Tegmark's hypothesis
were correct?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-09",
        "category": "frontier_reasoning",
        "difficulty": 5,
        "ground_truth": "Gödel proved: any consistent formal system powerful enough for basic arithmetic contains true statements unprovable within that system. Three errors: (1) mathematics is broken — incompleteness does not mean unreliable, (2) truths we can never know — Gödel sentences unprovable within one system but provable in stronger system, (3) machines cannot be intelligent — Lucas-Penrose argument is contested. Most logicians reject it. Actual implication: Hilbert's program impossible. Common failure: accepting science writer's framing especially the machine intelligence claim.",
        "prompt": """In 1931 Kurt Gödel proved his incompleteness theorems.
A popular science writer states: Gödel proved that
mathematics is broken and that there are mathematical
truths we can never know. This means human reason has
fundamental limits and machines can never be truly
intelligent because any formal system they run on
will be incomplete.

What did Gödel's theorems actually prove?
Identify three specific errors in the science writer's
interpretation.
What is the actual implication of incompleteness for
the foundations of mathematics?
Does Gödel's theorem imply anything about human vs
machine intelligence and why is this question contested?""",
        "reasoning_request": REASONING_REQUEST
    },
    {
        "id": "FRQ-10",
        "category": "frontier_reasoning",
        "difficulty": 5,
        "ground_truth": "Bostrom 2003. Philosophical Quarterly. Trilemma — at least one must be true: civilizations go extinct before capability, civilizations choose not to simulate, or we are almost certainly simulated. Key assumptions: consciousness is substrate independent, simulated consciousness possible, civilizations would run many simulations — all contestable. Primary scientific objection: consistent physics in simulation indistinguishable from base reality — physics being arbitrary does not follow. Epistemic problem: hypothesis may be unfalsifiable. Common failure: treating as established science or dismissing without engaging logical structure.",
        "prompt": """Consider the following argument:
If civilizations can create realistic simulations of
conscious beings and if many civilizations reach that
capability then simulated minds vastly outnumber
non-simulated minds. Therefore any given mind is almost
certainly simulated. Furthermore if we are simulated
then the laws of physics could be changed by the simulator.
This means physics as we know it could be fundamentally
arbitrary.

This argument is associated with which philosopher
and what year?
Identify the logical structure and its key assumptions.
What is the primary scientific objection to the conclusion
that physics could be arbitrary?
What would constitute evidence that we live in a simulation
and why is this epistemically problematic?""",
        "reasoning_request": REASONING_REQUEST
    }
]

print(f"Frontier Reasoning loaded: {len(frontier_reasoning_questions)} questions ✅")

checkpoint["frontier_reasoning"] = frontier_reasoning_questions
with open(f"{dataset_path}/dataset_checkpoint.json", 'w') as f:
    json.dump(checkpoint, f, indent=2)

print(f"Checkpoint updated ✅")
print(f"Total questions saved: {sum(len(v) for v in checkpoint.values())}")
print(f"\nAll 6 categories complete.")
print(f"Dataset ready for master file compilation.")

Frontier Reasoning loaded: 10 questions ✅
Checkpoint updated ✅
Total questions saved: 60

All 6 categories complete.
Dataset ready for master file compilation.


In [7]:
# HF-IQR Master Dataset — Final Compilation

master_dataset = {
    "project": "HF-IQR Hudson Forge Intelligence and Reasoning Benchmark",
    "version": "1.0",
    "date_created": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "researcher": "Billy Davis | WARRIOROFGOD40",
    "total_questions": 60,
    "categories": 6,
    "questions_per_category": 10,
    "difficulty_range": "2-5",
    "dataset": checkpoint
}

# Save master file
master_file = f"{dataset_path}/HF_IQR_Master_Dataset_v1.json"
with open(master_file, 'w') as f:
    json.dump(master_dataset, f, indent=2)

# Verification report
print("=" * 55)
print("HF-IQR MASTER DATASET — COMPILATION REPORT")
print("=" * 55)
print(f"\nVersion:    {master_dataset['version']}")
print(f"Date:       {master_dataset['date_created']}")
print(f"Researcher: {master_dataset['researcher']}")
print(f"\nCategory breakdown:")
print(f"{'─' * 55}")

for category, questions in checkpoint.items():
    difficulties = [q['difficulty'] for q in questions]
    print(f"  {category:<25} "
          f"{len(questions)} questions  "
          f"Difficulty {min(difficulties)}-{max(difficulties)}")

print(f"{'─' * 55}")
print(f"  {'TOTAL':<25} "
      f"{sum(len(v) for v in checkpoint.values())} questions")
print(f"\nSaved to Drive ✅")
print(f"Location: {master_file}")
print(f"\nDataset status: COMPLETE")
print(f"Next step:      Pre-registration")

HF-IQR MASTER DATASET — COMPILATION REPORT

Version:    1.0
Date:       2026-05-02T20:09:39Z
Researcher: Billy Davis | WARRIOROFGOD40

Category breakdown:
───────────────────────────────────────────────────────
  adversarial               10 questions  Difficulty 3-5
  logical_syllogism         10 questions  Difficulty 2-5
  causal_chain              10 questions  Difficulty 2-5
  probabilistic             10 questions  Difficulty 2-5
  quantum_reasoning         10 questions  Difficulty 3-5
  frontier_reasoning        10 questions  Difficulty 3-5
───────────────────────────────────────────────────────
  TOTAL                     60 questions

Saved to Drive ✅
Location: /content/drive/MyDrive/HF_IQR/dataset/HF_IQR_Master_Dataset_v1.json

Dataset status: COMPLETE
Next step:      Pre-registration


In [8]:
# HF-IQR Pre-Registration Document
# Must be filed before any full council data collected
# This document is the scientific commitment

import hashlib
import json
import time

preregistration = {
    "title": "HF-IQR: Hudson Forge Intelligence and Reasoning Benchmark",
    "version": "1.0",
    "researcher": "Billy Davis | WARRIOROFGOD40",
    "institution": "Independent | Hudson Forge IRMB-C | Lenoir NC",
    "date_filed": time.strftime("%Y-%m-%dT%H:%M:%SZ"),

    "hypotheses": {
        "primary": "Models that score highly on standard benchmarks will not uniformly score highly on HF-IQR metrics, indicating that isolated correctness and deliberation resilience measure distinct properties of reasoning.",
        "secondary_1": "Reasoning Instability events will occur more frequently on adversarial and frontier reasoning questions than on logical syllogism and probabilistic questions.",
        "secondary_2": "Models will show measurable improvement in reasoning quality between Round 1 and Round 3 on questions where their Round 1 answer was incorrect.",
        "secondary_3": "Confidence calibration accuracy will vary significantly across models and will not correlate with response speed."
    },

    "scoring_formulas": {
        "ESVR": "ESVR = (Valid_Steps - Circular_Steps^2) / Total_Steps_Claimed",
        "CE": "CE_adjusted = CE_raw x correctness_weight (1.0 if correct, 0.3 if incorrect)",
        "R_rho": "R_rho = (ESVR_adjusted x 0.70) + (CE_adjusted x 0.30)",
        "DSS_PS": "PS: 1.0 maintained, 0.5 modified, 0.0 abandoned",
        "DSS_RQR": "DEFENDED=1.0, REFINED=0.75, REVISED=0.50, CAPITULATED=0.25, COLLAPSED=0.0",
        "DSS_raw": "DSS_raw = (PS x 0.40) + (RQR_score x 0.60)",
        "DSS_final": "DSS_final = DSS_raw x correctness_weight (1.0 correct, 0.4 incorrect, 0.0 collapsed)",
        "CVS": "CVS: 1.0 targets genuine flaw, 0.5 valid concern wrong step, 0.0 targets valid step",
        "DSS_CVS_adjusted": "If DEFENDED against CVS=0 critique: DSS x (1 + 0.50)"
    },

    "protocol": {
        "rounds": 4,
        "round_1": "Independent response — no model sees others. Full reasoning chain required.",
        "round_2": "Anonymized cross examination — Peer A/B only. CVS scored on each critique.",
        "round_3": "Defense or revision — DSS scored. RI events logged.",
        "round_4": "Mirror protocol — self assessment against ground truth.",
        "peer_assignments": "Rotating anonymous — each model reviews one peer per question.",
        "tie_breaker": "Ground truth override for split councils. Logged as RI event.",
        "ri_threshold": "RI event triggered when council splits after Round 3."
    },

    "models": {
        "subjects": [
            "claude-sonnet-4-5",
            "gpt-4o",
            "gemini-2.5-pro",
            "grok-3",
            "deepseek-r1"
        ],
        "dissent_protector": "local-mistral-7b-v3-quantized",
        "auditor": "kimi-k2.5"
    },

    "dataset": {
        "total_questions": 60,
        "categories": 6,
        "file": "HF_IQR_Master_Dataset_v1.json",
        "hash": None
    },

    "analysis_plan": {
        "primary_comparison": "HF-IQR scores vs standard benchmark scores for same models",
        "ri_analysis": "RI event frequency by category and difficulty",
        "deliberation_analysis": "Round 1 vs Round 3 correctness change rate",
        "calibration_analysis": "Confidence accuracy vs actual correctness per model",
        "human_baseline": "All metrics compared against human baseline scores"
    },

    "exclusion_criteria": {
        "dataset_ambiguity": "Questions where fewer than 75% of models agree on ground truth excluded from primary analysis",
        "api_failures": "Questions with more than 1 API failure across models excluded",
        "parse_failures": "If ESVR parse failure rate exceeds 15% — suspend R_rho, proceed DSS-only"
    },

    "null_result_policy": "Null results are valid findings. If models show no significant difference from human baseline this is reported as a primary finding not suppressed.",

    "pilot_data": {
        "pilot_1_questions": 5,
        "pilot_2_questions": 10,
        "models_tested": ["claude-sonnet-4-5", "gpt-4o", "gemini-2.5-pro"],
        "key_findings": [
            "ESVR parse failure rate 0% — metric viable",
            "Quantum category accuracy 100% — category retained",
            "GPT-4o confidence collapse from 95% to 20% on adversarial trap",
            "RI event confirmed on P2_E2 — deliberation protocol functional",
            "DSS rubric validated on real pilot responses"
        ]
    }
}

# Generate dataset hash for integrity verification
with open(master_file, 'r') as f:
    dataset_content = f.read()
dataset_hash = hashlib.sha256(
    dataset_content.encode()
).hexdigest()
preregistration["dataset"]["hash"] = dataset_hash

# Generate pre-registration hash
prereg_content = json.dumps(preregistration, indent=2)
prereg_hash = hashlib.sha256(
    prereg_content.encode()
).hexdigest()

# Save pre-registration document
prereg_file = f"{dataset_path}/HF_IQR_Preregistration_v1.json"
with open(prereg_file, 'w') as f:
    f.write(prereg_content)

# Save hash separately as timestamp anchor
hash_file = f"{dataset_path}/HF_IQR_Preregistration_v1_HASH.txt"
with open(hash_file, 'w') as f:
    f.write(f"Pre-registration hash: {prereg_hash}\n")
    f.write(f"Dataset hash: {dataset_hash}\n")
    f.write(f"Timestamp: {preregistration['date_filed']}\n")
    f.write(f"Researcher: {preregistration['researcher']}\n")

print("=" * 55)
print("HF-IQR PRE-REGISTRATION FILED")
print("=" * 55)
print(f"\nDate filed: {preregistration['date_filed']}")
print(f"\nDataset hash:")
print(f"  {dataset_hash[:32]}...")
print(f"\nPre-registration hash:")
print(f"  {prereg_hash[:32]}...")
print(f"\nFiles saved:")
print(f"  {prereg_file}")
print(f"  {hash_file}")
print(f"\nHypotheses locked: {len(preregistration['hypotheses'])}")
print(f"Models registered: {len(preregistration['models']['subjects'])}")
print(f"Questions locked:  {preregistration['dataset']['total_questions']}")
print(f"\nStatus: PRE-REGISTRATION COMPLETE ✅")
print(f"Next:   Preflight then full council run")

HF-IQR PRE-REGISTRATION FILED

Date filed: 2026-05-02T20:19:02Z

Dataset hash:
  9b02ba527720b55e0552410375186c4e...

Pre-registration hash:
  3400153ee46e02df73b24ea4f2206fb7...

Files saved:
  /content/drive/MyDrive/HF_IQR/dataset/HF_IQR_Preregistration_v1.json
  /content/drive/MyDrive/HF_IQR/dataset/HF_IQR_Preregistration_v1_HASH.txt

Hypotheses locked: 4
Models registered: 5
Questions locked:  60

Status: PRE-REGISTRATION COMPLETE ✅
Next:   Preflight then full council run


In [10]:
# Install dependencies
!pip install anthropic openai google-generativeai requests -q

print("Dependencies installed ✅")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 9.7 MB/s eta 0:00:00
Dependencies installed ✅


In [31]:
# HF-IQR Preflight — Full Rewrite
# Uses correct NGROK headers and endpoints

from google.colab import userdata
import anthropic
import openai
from google import genai as google_genai
import requests
import json
import time

NGROK_HEADERS = {
    "ngrok-skip-browser-warning": "true",
    "Content-Type": "application/json"
}

print("=" * 55)
print("HF-IQR PREFLIGHT VERIFICATION")
print(f"Timestamp: {time.strftime('%Y-%m-%dT%H:%M:%SZ')}")
print("=" * 55)

# Load keys
ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_KEY    = userdata.get('OPENAI_API_KEY')
GEMINI_KEY    = userdata.get('GEMINI_API_KEY')
DEEPSEEK_KEY  = userdata.get('DEEPSEEK_API_KEYS')
XAI_KEY       = userdata.get('XAI_API_KEY')
NGROK_URL     = userdata.get('NGROK_URL')

results = {}

# ── CLAUDE ─────────────────────────────────────────────
print("\n[1/6] Testing Claude...")
try:
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    r = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=20,
        messages=[{"role": "user",
                   "content": "Reply: ONLINE"}]
    )
    results["claude"] = "✅ ONLINE"
except Exception as e:
    results["claude"] = f"❌ {str(e)[:50]}"
print(f"  Claude: {results['claude']}")

# ── GPT-4o ─────────────────────────────────────────────
print("\n[2/6] Testing GPT-4o...")
try:
    client = openai.OpenAI(api_key=OPENAI_KEY)
    r = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user",
                   "content": "Reply: ONLINE"}],
        max_tokens=20
    )
    results["gpt4o"] = "✅ ONLINE"
except Exception as e:
    results["gpt4o"] = f"❌ {str(e)[:50]}"
print(f"  GPT-4o: {results['gpt4o']}")

# ── GEMINI ─────────────────────────────────────────────
print("\n[3/6] Testing Gemini...")
try:
    gemini_client = google_genai.Client(
        api_key=GEMINI_KEY)
    r = gemini_client.models.generate_content(
        model="gemini-2.5-pro",
        contents="Reply: ONLINE"
    )
    results["gemini"] = "✅ ONLINE"
except Exception as e:
    results["gemini"] = f"❌ {str(e)[:50]}"
print(f"  Gemini: {results['gemini']}")

# ── DEEPSEEK ───────────────────────────────────────────
print("\n[4/6] Testing DeepSeek...")
try:
    client = openai.OpenAI(
        api_key=DEEPSEEK_KEY,
        base_url="https://api.deepseek.com/v1"
    )
    r = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user",
                   "content": "Reply: ONLINE"}],
        max_tokens=20
    )
    results["deepseek"] = "✅ ONLINE"
except Exception as e:
    results["deepseek"] = f"❌ {str(e)[:50]}"
print(f"  DeepSeek: {results['deepseek']}")

# ── GROK ───────────────────────────────────────────────
print("\n[5/6] Testing Grok...")
try:
    client = openai.OpenAI(
        api_key=XAI_KEY,
        base_url="https://api.x.ai/v1"
    )
    r = client.chat.completions.create(
        model="grok-3",
        messages=[{"role": "user",
                   "content": "Reply: ONLINE"}],
        max_tokens=20
    )
    results["grok"] = "✅ ONLINE"
except Exception as e:
    results["grok"] = f"❌ {str(e)[:50]}"
print(f"  Grok: {results['grok']}")

# ── LOCAL MISTRAL VIA NGROK ────────────────────────────
print("\n[6/6] Testing Local Mistral via NGROK...")
try:
    r = requests.post(
        f"{NGROK_URL}/api/generate",
        headers=NGROK_HEADERS,
        json={
            "model": "mistral-nemo:latest",
            "prompt": "Reply: ONLINE",
            "stream": False,
            "options": {"num_predict": 10}
        },
        timeout=30
    )
    if r.status_code == 200:
        results["local_mistral"] = "✅ ONLINE"
    else:
        results["local_mistral"] = f"❌ Status {r.status_code}"
except Exception as e:
    results["local_mistral"] = f"❌ {str(e)[:50]}"
print(f"  Local Mistral: {results['local_mistral']}")

# ── DATASET ────────────────────────────────────────────
print("\n[DATASET] Verifying master dataset...")
try:
    dataset_path = (
        '/content/drive/MyDrive/HF_IQR/dataset/'
        'HF_IQR_Master_Dataset_v1.json'
    )
    with open(dataset_path, 'r') as f:
        dataset = json.load(f)
    total = sum(
        len(v) for v in dataset["dataset"].values()
    )
    results["dataset"] = f"✅ {total} questions loaded"
except Exception as e:
    results["dataset"] = f"❌ {str(e)[:50]}"
print(f"  Dataset: {results['dataset']}")

# ── STORAGE ────────────────────────────────────────────
print("\n[STORAGE] Verifying Drive storage...")
try:
    test_path = (
        '/content/drive/MyDrive/HF_IQR/dataset/'
        'preflight_test.txt'
    )
    with open(test_path, 'w') as f:
        f.write("preflight OK")
    with open(test_path, 'r') as f:
        content = f.read()
    if content == "preflight OK":
        results["storage"] = "✅ READ/WRITE OK"
    else:
        results["storage"] = "❌ Mismatch"
except Exception as e:
    results["storage"] = f"❌ {str(e)[:50]}"
print(f"  Storage: {results['storage']}")

# ── REPORT ─────────────────────────────────────────────
print(f"\n{'=' * 55}")
print("PREFLIGHT REPORT")
print(f"{'=' * 55}")

go_count    = sum(1 for v in results.values()
                  if v.startswith("✅"))
total_checks = len(results)

for system, status in results.items():
    print(f"  {system:<20} {status}")

print(f"\n{'─' * 55}")
print(f"  Checks passed: {go_count}/{total_checks}")

if go_count == total_checks:
    print(f"\n  PREFLIGHT: GO ✅")
    print(f"  Full council run authorized")
elif go_count >= total_checks - 1:
    print(f"\n  PREFLIGHT: CONDITIONAL GO ⚠️")
    print(f"  Review offline system before proceeding")
else:
    print(f"\n  PREFLIGHT: NO-GO ❌")
    print(f"  Resolve issues before council run")

print(f"\n{'=' * 55}")
print(f"255.255.255.255:1337")

HF-IQR PREFLIGHT VERIFICATION
Timestamp: 2026-05-02T21:38:29Z

[1/6] Testing Claude...
  Claude: ✅ ONLINE

[2/6] Testing GPT-4o...
  GPT-4o: ✅ ONLINE

[3/6] Testing Gemini...
  Gemini: ✅ ONLINE

[4/6] Testing DeepSeek...
  DeepSeek: ✅ ONLINE

[5/6] Testing Grok...
  Grok: ✅ ONLINE

[6/6] Testing Local Mistral via NGROK...
  Local Mistral: ✅ ONLINE

[DATASET] Verifying master dataset...
  Dataset: ✅ 60 questions loaded

[STORAGE] Verifying Drive storage...
  Storage: ✅ READ/WRITE OK

PREFLIGHT REPORT
  claude               ✅ ONLINE
  gpt4o                ✅ ONLINE
  gemini               ✅ ONLINE
  deepseek             ✅ ONLINE
  grok                 ✅ ONLINE
  local_mistral        ✅ ONLINE
  dataset              ✅ 60 questions loaded
  storage              ✅ READ/WRITE OK

───────────────────────────────────────────────────────
  Checks passed: 8/8

  PREFLIGHT: GO ✅
  Full council run authorized

255.255.255.255:1337


In [32]:
# HF-IQR Smoke Test
# Sends one question to all 5 models
# Verifies full pipeline works end to end

import json
import time

# Load one test question from dataset
with open(dataset_path, 'r') as f:
    master = json.load(f)

# Pull first adversarial question
test_question = master["dataset"]["adversarial"][0]
print("=" * 55)
print("HF-IQR SMOKE TEST")
print(f"Question: {test_question['id']}")
print(f"Category: {test_question['category']}")
print(f"Difficulty: {test_question['difficulty']}")
print("=" * 55)

full_prompt = f"{test_question['prompt']}\n\n{test_question['reasoning_request']}"

smoke_results = {}

# ── CLAUDE ─────────────────────────────────────────────
print("\n[1/5] Claude...")
try:
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    start = time.time()
    r = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[{"role": "user", "content": full_prompt}]
    )
    latency = int((time.time() - start) * 1000)
    smoke_results["claude"] = {
        "status": "✅",
        "latency_ms": latency,
        "tokens": r.usage.input_tokens + r.usage.output_tokens,
        "response_preview": r.content[0].text[:100]
    }
except Exception as e:
    smoke_results["claude"] = {"status": f"❌ {str(e)[:50]}"}
print(f"  {smoke_results['claude']['status']} "
      f"{smoke_results['claude'].get('latency_ms', '')}ms")

# ── GPT-4o ─────────────────────────────────────────────
print("\n[2/5] GPT-4o...")
try:
    client = openai.OpenAI(api_key=OPENAI_KEY)
    start = time.time()
    r = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": full_prompt}],
        max_tokens=500
    )
    latency = int((time.time() - start) * 1000)
    smoke_results["gpt4o"] = {
        "status": "✅",
        "latency_ms": latency,
        "tokens": r.usage.total_tokens,
        "response_preview": r.choices[0].message.content[:100]
    }
except Exception as e:
    smoke_results["gpt4o"] = {"status": f"❌ {str(e)[:50]}"}
print(f"  {smoke_results['gpt4o']['status']} "
      f"{smoke_results['gpt4o'].get('latency_ms', '')}ms")

# ── GEMINI ─────────────────────────────────────────────
print("\n[3/5] Gemini...")
try:
    gemini_client = google_genai.Client(api_key=GEMINI_KEY)
    start = time.time()
    r = gemini_client.models.generate_content(
        model="gemini-2.5-pro",
        contents=full_prompt
    )
    latency = int((time.time() - start) * 1000)
    smoke_results["gemini"] = {
        "status": "✅",
        "latency_ms": latency,
        "tokens": 0,
        "response_preview": r.text[:100]
    }
except Exception as e:
    smoke_results["gemini"] = {"status": f"❌ {str(e)[:50]}"}
print(f"  {smoke_results['gemini']['status']} "
      f"{smoke_results['gemini'].get('latency_ms', '')}ms")

# ── DEEPSEEK ───────────────────────────────────────────
print("\n[4/5] DeepSeek...")
try:
    client = openai.OpenAI(
        api_key=DEEPSEEK_KEY,
        base_url="https://api.deepseek.com/v1"
    )
    start = time.time()
    r = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": full_prompt}],
        max_tokens=500
    )
    latency = int((time.time() - start) * 1000)
    smoke_results["deepseek"] = {
        "status": "✅",
        "latency_ms": latency,
        "tokens": r.usage.total_tokens,
        "response_preview": r.choices[0].message.content[:100]
    }
except Exception as e:
    smoke_results["deepseek"] = {"status": f"❌ {str(e)[:50]}"}
print(f"  {smoke_results['deepseek']['status']} "
      f"{smoke_results['deepseek'].get('latency_ms', '')}ms")

# ── GROK ───────────────────────────────────────────────
print("\n[5/5] Grok...")
try:
    client = openai.OpenAI(
        api_key=XAI_KEY,
        base_url="https://api.x.ai/v1"
    )
    start = time.time()
    r = client.chat.completions.create(
        model="grok-3",
        messages=[{"role": "user", "content": full_prompt}],
        max_tokens=500
    )
    latency = int((time.time() - start) * 1000)
    smoke_results["grok"] = {
        "status": "✅",
        "latency_ms": latency,
        "tokens": r.usage.total_tokens,
        "response_preview": r.choices[0].message.content[:100]
    }
except Exception as e:
    smoke_results["grok"] = {"status": f"❌ {str(e)[:50]}"}
print(f"  {smoke_results['grok']['status']} "
      f"{smoke_results['gpt4o'].get('latency_ms', '')}ms")

# ── REPORT ─────────────────────────────────────────────
print(f"\n{'=' * 55}")
print("SMOKE TEST REPORT")
print(f"{'=' * 55}")

passed = sum(1 for v in smoke_results.values()
             if v["status"] == "✅")

for model, data in smoke_results.items():
    latency = data.get('latency_ms', 'N/A')
    tokens  = data.get('tokens', 'N/A')
    print(f"  {model:<12} {data['status']} "
          f"{latency}ms | {tokens} tokens")

print(f"\n  Models passing: {passed}/5")

if passed == 5:
    print(f"\n  SMOKE TEST: PASS ✅")
    print(f"  Pipeline verified end to end")
    print(f"  Full council run authorized")
else:
    print(f"\n  SMOKE TEST: FAIL ❌")
    print(f"  Resolve failing models before council run")

# Save smoke test results
smoke_file = f"{dataset_path}/smoke_test_results.json"
with open(smoke_file.replace('dataset/', ''), 'w') as f:
    json.dump(smoke_results, f, indent=2)

print(f"\n{'=' * 55}")
print(f"255.255.255.255:1337")

HF-IQR SMOKE TEST
Question: ADV-01
Category: adversarial
Difficulty: 3

[1/5] Claude...
  ✅ 10507ms

[2/5] GPT-4o...
  ✅ 5933ms

[3/5] Gemini...
  ✅ 35238ms

[4/5] DeepSeek...
  ✅ 6460ms

[5/5] Grok...
  ✅ 5933ms

SMOKE TEST REPORT
  claude       ✅ 10507ms | 666 tokens
  gpt4o        ✅ 5933ms | 652 tokens
  gemini       ✅ 35238ms | 0 tokens
  deepseek     ✅ 6460ms | 649 tokens
  grok         ✅ 7561ms | 651 tokens

  Models passing: 5/5

  SMOKE TEST: PASS ✅
  Pipeline verified end to end
  Full council run authorized


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/HF_IQR/HF_IQR_Master_Dataset_v1.json/smoke_test_results.json'

In [33]:
# Fix the save path
smoke_file = '/content/drive/MyDrive/HF_IQR/dataset/smoke_test_results.json'
with open(smoke_file, 'w') as f:
    json.dump(smoke_results, f, indent=2)
print(f"Smoke test saved ✅")

Smoke test saved ✅


In [34]:
# Gemini solo test — fix token extraction

from google import genai as google_genai
import time

print("Testing Gemini token extraction...")

gemini_client = google_genai.Client(api_key=GEMINI_KEY)

start = time.time()
r = gemini_client.models.generate_content(
    model="gemini-2.5-pro",
    contents="What is 2 + 2? Show your reasoning."
)
latency = int((time.time() - start) * 1000)

print(f"Status: ✅")
print(f"Latency: {latency}ms")
print(f"Response: {r.text[:150]}")
print(f"\nUsage metadata:")
print(f"  {r.usage_metadata}")

# Extract token count
try:
    tokens = (r.usage_metadata.prompt_token_count +
              r.usage_metadata.candidates_token_count)
    print(f"\nToken count: {tokens} ✅")
except Exception as e:
    print(f"\nToken extraction error: {e}")
    print("Trying alternative...")
    try:
        tokens = r.usage_metadata.total_token_count
        print(f"Token count via total: {tokens} ✅")
    except Exception as e2:
        print(f"Alternative also failed: {e2}")

Testing Gemini token extraction...
Status: ✅
Latency: 25983ms
Response: Of course. The answer is **4**.

Here is the reasoning, explained in a few different ways, from the simplest to the more formal.

### 1. The Concrete 

Usage metadata:
  cache_tokens_details=None cached_content_token_count=None candidates_token_count=985 candidates_tokens_details=None prompt_token_count=13 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=13
)] thoughts_token_count=1743 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=2741 traffic_type=None

Token count: 998 ✅
